In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [8]:
from sklearn.model_selection import train_test_split

# Investigating data

In [ ]:
df = pd.read_csv("../data/balanced_ai_human_prompts.csv")

In [3]:
(df['generated'] == 0).sum()

np.int64(1375)

In [4]:
(df['generated'] == 1).sum()

np.int64(1375)

=> equal number of data for both sides 

In [5]:
df.tail()

,text,generated
2745,Generate a detailed summary of global healthca...,1
2746,Compose an in-depth exploration of financial t...,1
2747,Generate a detailed summary of autonomous vehi...,1
2748,Develop a persuasive argument about internet o...,1
2749,Generate a detailed summary of supply chain ma...,1


In [6]:
df = df.sample(frac=1).reset_index(drop=True)
df.tail()

,text,generated
2745,Produce a balanced review of cultural anthropo...,1
2746,The electoral college is a bad thing because v...,0
2747,"Every four years, the United States is turned ...",0
2748,Produce a balanced review of blockchain applic...,1
2749,"Many people throughout the world, would agree ...",0


In [7]:
df['text'].str.len().quantile(0.95)

np.float64(4404.749999999998)

In [9]:
# dividng train test
X = np.array(df['text'].values)
y = np.array(df['generated'].values)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train: (2200,)
X_test: (550,)
y_train: (2200,)
y_test: (550,)


# Tokenize

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [18]:
vocab_size = 16000
max_len = 4405
embedding_dim = 128

In [19]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [20]:
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')

X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# Train

In [21]:
model = keras.Sequential([
        keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=embedding_dim,
            input_length=max_len,
            name="embedding_5"
        ),
        keras.layers.GlobalAveragePooling1D(),
        keras.layers.Dense(64, activation="relu", name="dense_10"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid", name="dense_11"),
    ])

model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
)

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
history = model.fit(
    X_train_pad, y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.5449 - loss: 0.6897 - val_accuracy: 0.5114 - val_loss: 0.6589
Epoch 2/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.6778 - loss: 0.6260 - val_accuracy: 0.6295 - val_loss: 0.5626
Epoch 3/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.8114 - loss: 0.4554 - val_accuracy: 0.9841 - val_loss: 0.3460
Epoch 4/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.9364 - loss: 0.2353 - val_accuracy: 0.9909 - val_loss: 0.1090
Epoch 5/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step - accuracy: 0.9864 - loss: 0.1011 - val_accuracy: 0.9886 - val_loss: 0.0558
Epoch 6/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.9903 - loss: 0.0603 - val_accuracy: 0.9886 - val_loss: 0.0371
Epoch 7/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.9903 - loss: 0.0399 - val_accuracy: 0.9932 - val_loss: 0.0263
Epoch 8/10
55/55 ━━━━━━━━━━━━━━━━━━━━ 4s 66ms/step - accuracy: 0.9773 - loss: 0.0718 - val_accuracy: 0.9886 - v

In [25]:
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Accuracy: {acc:.4f}")

Test Accuracy: 0.9982


In [28]:
model.save('alp.h5')

# Predictions

In [30]:
loaded_model = keras.models.load_model('alp.h5')

In [52]:
text_sample = [
    X_test[3],
    'Access to quality education has long been regarded as a fundamental human right, yet millions of children around the world remain excluded from meaningful learning opportunities. In the twenty-first century, this exclusion has taken on a new dimension: the digital divide. As classrooms increasingly rely on technology — from online learning platforms to digital textbooks — students without reliable internet access or devices are being left further behind. While wealthier schools equip students with the tools needed to thrive in a technology-driven world, those in low-income or rural communities struggle to keep pace. Bridging this gap is no longer a matter of convenience but of equity, and without deliberate action, digital inequality threatens to deepen the very educational disparities that universal education policies seek to eliminate.'
]
input_sample = np.array(text_sample)
input_sample = tokenizer.texts_to_sequences(input_sample)
input_sample = pad_sequences(input_sample, maxlen=max_len, padding='post', truncating='post')

In [53]:
predictions = loaded_model.predict(input_sample)
class_labels = (predictions > 0.5).astype(int)
class_labels

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step


array([[0],
       [1]])